In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. SYSTEM CONTEXT & CONSTANTS ---
k = 1.0  # Spring stiffness
L0 = 1.5  # Relaxed spring length (L0 > h creates bistability)
h = 1.0  # Anchor distance from origin along y-axis
m = 1.0  # Node mass
c = 0.1  # Damping coefficient (to see energy settle into a well)

# Fixed Anchor Coordinates
anchor_top = np.array([0.0, h])
anchor_bottom = np.array([0.0, -h])


# --- 2. FORCE & POTENTIAL CALCULATORS ---
def calculate_state(pos):
    """Computes total potential energy and net force vector at a given 2D position."""
    v_top = pos - anchor_top
    v_bot = pos - anchor_bottom

    L1 = np.linalg.norm(v_top)
    L2 = np.linalg.norm(v_bot)

    # Total potential energy (scalar sum)
    pe = 0.5 * k * (L1 - L0) ** 2 + 0.5 * k * (L2 - L0) ** 2

    # Net Hookean spring forces
    # Avoid division by zero at anchor positions
    f_top = -k * (L1 - L0) * (v_top / L1) if L1 > 1e-9 else np.zeros(2)
    f_bot = -k * (L2 - L0) * (v_bot / L2) if L2 > 1e-9 else np.zeros(2)

    net_force = f_top + f_bot
    return pe, net_force


# --- 3. NUMERICAL INTEGRATION LOOP ---
dt = 0.05
steps = 400

# Initial conditions (place mass near the center barrier with an initial push)
pos = np.array([-0.2, 0.1])
vel = np.array([1.5, 0.0])

# Storage arrays for plotting
history_pos = []
history_pe = []
time_vals = np.arange(steps) * dt

# Initialize first step
pe, force = calculate_state(pos)

for _ in range(steps):
    history_pos.append(pos.copy())
    history_pe.append(pe)

    # Velocity Verlet Update with structural damping (F_damping = -c * vel)
    f_total = force - c * vel
    vel_half = vel + (f_total / (2 * m)) * dt
    pos += vel_half * dt

    # Re-evaluate force and potential at new position
    pe, force = calculate_state(pos)

    f_total_new = force - c * vel_half  # Simple damping approximation
    vel = vel_half + (f_total_new / (2 * m)) * dt

history_pos = np.array(history_pos)

# --- 4. PLOTLY SUBPLOT VISUALIZATION ---
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "2D Trajectory Across Wells",
        "Potential Energy Extracted from Movement",
    ),
)

# Left Plot: 2D movement trajectory
fig.add_trace(
    go.Scatter(
        x=history_pos[:, 0],
        y=history_pos[:, 1],
        mode="lines+markers",
        name="Mass Path",
        marker=dict(size=4, color=history_pe, colorscale="Viridis", showscale=False),
        line=dict(color="gray", width=1),
    ),
    row=1,
    col=1,
)

# Highlight anchors and final resting position
fig.add_trace(
    go.Scatter(
        x=[0, 0],
        y=[h, -h],
        mode="markers",
        name="Anchors",
        marker=dict(color="red", size=10, symbol="square"),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=[history_pos[-1, 0]],
        y=[history_pos[-1, 1]],
        mode="markers",
        name="Settled Well",
        marker=dict(color="blue", size=12),
    ),
    row=1,
    col=1,
)

# Right Plot: Extracted potential energy over time sequence
fig.add_trace(
    go.Scatter(
        x=time_vals,
        y=history_pe,
        mode="lines",
        name="PE(t)",
        line=dict(color="#10b981", width=3),
    ),
    row=1,
    col=2,
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    title_text="Numerical Spring Simulation Framework",
)
fig.update_xaxes(title_text="X Coordinate", row=1, col=1)
fig.update_yaxes(title_text="Y Coordinate", row=1, col=1)
fig.update_xaxes(title_text="Simulation Time (s)", row=1, col=2)
fig.update_yaxes(title_text="Potential Energy (J)", row=1, col=2)

fig.show()